# ZeroMRG Gate4 — one frozen training-image diagnostic

Prepared locally; **not submitted or executed**. Owner approval is required before running this notebook on Kaggle. No training or metrics. Exactly one fixed image from the original checkpoint training CSV; no validation/test references are opened. Epoch90 is the latest verified physical model, with missing historical85 disclosed.

Existing private dataset attachments supply COV-CTR and checkpoint backup. The independently verified epoch90 model/prompt bundle comes from the existing private baseline kernel output, SHA-pinned; no new dataset publication. If mount/runtime/privacy/quota or pins differ, stop for review.

Image → frozen official OpenAI CLIP ViT-L/14 → raw500 prompt Eq6 attention → Eq5 residual/L2 → historical six-layer Transformer → greedy report, EOS or max160 including BOS. The paper states equations/freezing/500 prompts/EOS-or-limit; decoder details and greedy choice preserve the producing implementation. Historical encoder/cache revision is unverified. No M-CLIP re-encoding or text-encoder download is needed for the fixed bank.

Worker hard limit1800 seconds, dependency command limit300 seconds, heartbeat30 seconds; these are engineering safeguards, not measured runtime or GPU-budget predictions. Failure preserves logs/JSON, kills the worker group, and never retries. Source/config/checkpoint/encoder/dependency fingerprints and a small diagnostic JSON are saved in a unique output directory. Fluent text does not establish clinical correctness or reproduction of Table1.

In [ ]:
import hashlib, json, sys, uuid
from pathlib import Path
SOURCES = {'zeromrg_gate1_runtime.py': '"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.\n\nThis profile preserves the recorded COV-CTR experiment, not modular defaults.\nNo training, model download, or inference is performed on import.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,\n                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,\n                    activation="gelu", norm_first=False, final_layer_norm=False,\n                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)\nTRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,\n                gradient_clip=5.0, patience=50, max_epochs=200,\n                precision="float32", scheduler=None,\n                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")\nMCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"\nDEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",\n                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",\n                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_checkpoint(path):\n    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")\n    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\nclass CheckpointReportDecoder(nn.Module):\n    def __init__(self, architecture=None):\n        super().__init__()\n        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)\n        a = self.architecture\n        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],\n                "This class implements the recorded Kaggle decoder only")\n        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])\n        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])\n        layer = nn.TransformerDecoderLayer(\n            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],\n            dropout=a["dropout"], activation=a["activation"], batch_first=True,\n            norm_first=a["norm_first"])\n        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])\n        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])\n\n    def forward(self, condition, input_ids):\n        a = self.architecture\n        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],\n                "Decoder IDs must be [B,T] within the saved positional limit")\n        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),\n                "Condition dimensions incompatible with checkpoint decoder")\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)\n        target = self.token_embedding(input_ids) + self.position_embedding(positions)\n        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],\n                                    dtype=torch.bool, device=input_ids.device), diagonal=1)\n        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),\n                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))\n        return self.output_projection(hidden)\n\n    @torch.no_grad()\n    def generate(self, condition, max_length=None):\n        self.eval()\n        a = self.architecture\n        limit = a["max_tokens"] if max_length is None else int(max_length)\n        require(2 <= limit <= a["max_tokens"], "Invalid generation length")\n        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)\n        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)\n        for _ in range(limit - 1):\n            next_id = self(condition, ids)[:, -1].argmax(-1)\n            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)\n            ids = torch.cat((ids, next_id[:, None]), dim=1)\n            finished |= next_id.eq(a["eos_id"])\n            if bool(finished.all()):\n                break\n        return ids\n\n\ndef validate_model_state(checkpoint, model):\n    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")\n    state = checkpoint.get("decoder")\n    require(isinstance(state, dict), "Required decoder state dictionary absent")\n    expected = model.state_dict()\n    require(set(state) == set(expected),\n            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")\n    for key, tensor in state.items():\n        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,\n                f"Decoder shape mismatch: {key}")\n        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")\n        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")\n    if "architecture" in checkpoint:\n        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")\n    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),\n                                ("max_tokens", model.architecture["max_tokens"])):\n        if key in checkpoint:\n            require(checkpoint[key] == expected_value, f"Saved {key} differs")\n\n\ndef inspect_training_state(checkpoint, model, best_checkpoint=None):\n    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")\n    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}\n    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")\n    epoch = checkpoint["epoch"]\n    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")\n    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},\n            "Unknown scheduler state; supply its verified implementation before resuming")\n    if "training_config" in checkpoint:\n        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")\n    history = checkpoint["history"]\n    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")\n    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()\n                for row in history), "History record fields absent or invalid")\n    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")\n    for row in history:\n        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),\n                "Non-finite history value")\n    selected = max(history, key=lambda row: row["validation_bleu4"])\n    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),\n                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")\n    require(type(checkpoint["epochs_without_improvement"]) is int and\n            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")\n    validate_model_state(checkpoint, model)\n    if best_checkpoint is not None:\n        validate_selected_best(checkpoint, model, best_checkpoint)\n    opt = checkpoint["optimizer"]\n    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")\n    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")\n    group = opt["param_groups"][0]\n    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,\n                              nesterov=False, maximize=False).items():\n        require(group.get(key, False if key == "maximize" else None) == expected,\n                f"SGD setting mismatch: {key}")\n    parameters = list(model.named_parameters())\n    saved_ids = group["params"]\n    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),\n            "Optimizer parameter count or mapping incompatible")\n    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")\n    nonzero = 0\n    for pid, (name, parameter) in zip(saved_ids, parameters):\n        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")\n        buffer = opt["state"][pid]["momentum_buffer"]\n        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and\n                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")\n        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")\n        nonzero += int(bool(buffer.ne(0).any()))\n    require(nonzero > 0, "All SGD momentum is zero")\n    if "parameter_names" in checkpoint:\n        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")\n    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],\n                best_validation_bleu4=float(selected["validation_bleu4"]),\n                epochs_without_improvement=checkpoint["epochs_without_improvement"],\n                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,\n                parameter_count=sum(p.numel() for _, p in parameters),\n                legacy_rng_missing="rng_state" not in checkpoint,\n                scheduler="none", compatibility="PASS")\n\n\ndef validate_selected_best(checkpoint, model, best_checkpoint):\n    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")\n    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])\n    require(best_checkpoint.get("epoch") == selected["epoch"] and\n            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),\n                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),\n            f"Best checkpoint disagrees with history: need epoch {selected[\'epoch\']}, "\n            f"found epoch {best_checkpoint.get(\'epoch\')}. Recover matching weights; do not substitute latest weights.")\n    validate_model_state(best_checkpoint, model)\n\n\ndef restore_weights_optimizer(checkpoint, model, device="cpu"):\n    """Restore latest state for diagnostics; not authorization to continue training."""\n    result = inspect_training_state(checkpoint, model)\n    model.load_state_dict(checkpoint["decoder"], strict=True)\n    model.to(device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)\n    optimizer.load_state_dict(checkpoint["optimizer"])\n    for parameter, state in optimizer.state.items():\n        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)\n    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):\n        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),\n                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),\n                "Momentum changed during restoration")\n    model.eval()\n    return optimizer, result\n\n\ndef restore_training(checkpoint, model, best_checkpoint, device="cpu"):\n    result = inspect_training_state(checkpoint, model, best_checkpoint)\n    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)\n    return optimizer, result\n\n\ndef read_manifest(directory, name):\n    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef validate_artifacts(directory, expected_hashes=None):\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}\n    if expected_hashes is not None:\n        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")\n    records = {name: read_manifest(directory, name + "_manifest.csv")\n               for name in ("train", "validation", "test", "prompt", "validated")}\n    ids = {}\n    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():\n        rows = records[name]\n        ids[name] = [row["image_id"] for row in rows]\n        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")\n    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))\n    require(not (train & val or train & test or val & test), "Split leakage")\n    require(train | val | test == set(ids["validated"]), "Split population differs")\n    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")\n    by_id = {r["image_id"]: r for r in records["validated"]}\n    for name in ("train", "validation", "test", "prompt"):\n        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),\n                f"{name} reports differ from validated manifest")\n    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))\n    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")\n    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,\n            "Vocabulary or positional capacity differs")\n    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")\n    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({\n        token for row in records["train"] for token in re.findall(r"\\w+|[^\\w\\s]", row["report"])})\n    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")\n    text = load_checkpoint(directory / "mclip_text_features.pt")\n    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")\n    for name in ("train", "validation"):\n        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")\n        for vector in text[name].values():\n            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and\n                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")\n    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")\n    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")\n    vectors = prompt["embeddings"]\n    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")\n    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])\n    # Separate encodings are not bitwise equal. Record the observed difference;\n    # membership and frozen-cache hashes above are the provenance assertions.\n    normalized_prompts = F.normalize(vectors, dim=-1)\n    cosine = (normalized_prompts * aligned).sum(-1)\n    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),\n                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),\n                            exact_encoder_revision="UNVERIFIED")\n    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,\n                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)\n\n\ndef encode_report(text, vocabulary):\n    lookup = {token: i for i, token in enumerate(vocabulary)}\n    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\\w+|[^\\w\\s]", text)] + [2]\n    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")\n    return ids\n\n\ndef decode_report(token_ids, vocabulary):\n    tokens = []\n    for raw in token_ids:\n        token = int(raw)\n        if token == 2:\n            break\n        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")\n        if token not in {0, 1}:\n            tokens.append(vocabulary[token])\n    text = re.sub(r"\\s+([.,!?;:%)\\]])", r"\\1", " ".join(tokens))\n    return re.sub(r"([(\\[])\\s+", r"\\1", text).strip()\n\n\ndef enrich(image, prompts):\n    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,\n            "Image/prompt dimensions must share the recorded 768-D CLIP space")\n    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")\n    scores = image @ prompts.T / math.sqrt(768)\n    attention = scores.softmax(-1)\n    residual = image + attention @ prompts\n    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")\n    phi = F.normalize(residual, p=2, dim=-1)\n    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")\n    return phi, attention\n\n\ndef capture_rng():\n    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),\n                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    torch.set_rng_state(state["torch_cpu"].cpu())\n    if state["torch_cuda"]:\n        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")\n        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])\n\n\ndef atomic_save(payload, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, destination)\n\n\ndef training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):\n    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),\n                optimizer=optimizer.state_dict(), scheduler=None, history=history,\n                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,\n                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],\n                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,\n                rng_state=capture_rng())\n', 'zeromrg_gate4_runtime.py': '"""Frozen COV-CTR Gate4 inference; no loading, downloading or execution on import."""\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom zeromrg_gate1_runtime import (\n    ARCHITECTURE, MCLIP_NAME, CheckpointReportDecoder, decode_report, enrich,\n    load_checkpoint, require, sha256, validate_model_state,\n)\n\nCLIP_COMMIT = "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"\nCLIP_SOURCE_SHA256 = "9540f200fbf8145479fa655382a56dab048d238cc698b9cbd8df3b6d86d3f1b6"\nCLIP_WEIGHT_SHA256 = "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836"\nEPOCH90_SHA256 = "6166e36198737bfbe9e0c35f03f0b320c44e76f3e05068ef6299d74e9179c6b2"\nINFERENCE_FILES = (\n    "train_manifest.csv", "prompt_manifest.csv", "decoder_vocabulary.json", "mclip_prompt_bank.pt",\n)\n\n\ndef freeze(module):\n    module.eval()\n    module.requires_grad_(False)\n    require(all(p.grad is None for p in module.parameters()), "Unexpected existing gradients")\n    return module\n\n\ndef assert_frozen(module):\n    require(not module.training and all(not p.requires_grad and p.grad is None\n            for p in module.parameters()), "Inference module is not frozen/eval")\n\n\ndef load_inference_artifacts(directory, expected_hashes):\n    """Read train-only inputs. Never open validation/test references or text cache."""\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in INFERENCE_FILES}\n    require(hashes == {name: expected_hashes[name] for name in INFERENCE_FILES},\n            "Inference artifact hashes differ from verified checkpoint family")\n    def ids(name):\n        with (directory / name).open(encoding="utf-8-sig", newline="") as handle:\n            return [row["image_id"] for row in csv.DictReader(handle)]\n    train_ids, prompt_ids = ids("train_manifest.csv"), ids("prompt_manifest.csv")\n    require(len(train_ids) == len(set(train_ids)) == 571, "Training IDs differ")\n    require(len(prompt_ids) == len(set(prompt_ids)) == 500 and\n            set(prompt_ids) <= set(train_ids), "Prompt membership is not training-only")\n    bank = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    validate_prompt_bank(bank, prompt_ids)\n    vocab = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocab["max_tokens"] == 160 and len(vocab["vocabulary"]) == 368 and\n            vocab["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Vocabulary differs")\n    require(vocab["vocabulary"][:4] == ["<pad>", "<bos>", "<eos>", "<unk>"], "Token IDs differ")\n    return dict(train_ids=train_ids, prompt_ids=prompt_ids, prompts=bank["embeddings"].detach(),\n                vocabulary=vocab["vocabulary"], hashes=hashes)\n\n\ndef validate_prompt_bank(bank, prompt_ids):\n    require(bank["model"] == MCLIP_NAME and bank.get("normalization") == "raw",\n            "Wrong prompt encoder or normalization")\n    require(bank["prompt_ids"] == prompt_ids, "Prompt bank row ordering differs")\n    vectors = bank["embeddings"]\n    require(tuple(vectors.shape) == (500, 768) and vectors.dtype == torch.float32 and\n            not vectors.requires_grad and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid fixed raw prompt bank")\n\n\ndef restore_epoch90(path, device="cpu", expected_sha256=EPOCH90_SHA256):\n    require(sha256(path) == expected_sha256, "Epoch90 checkpoint hash differs")\n    checkpoint = load_checkpoint(path)\n    require(checkpoint.get("epoch") == 90, "Diagnostic requires verified physical epoch90")\n    with torch.device("meta"):\n        decoder = CheckpointReportDecoder()\n    validate_model_state(checkpoint, decoder)\n    decoder.load_state_dict(checkpoint["decoder"], strict=True, assign=True)\n    freeze(decoder)\n    decoder.to(device)\n    require(decoder.architecture == ARCHITECTURE, "Historical decoder architecture differs")\n    return decoder\n\n\ndef load_companion_clip(device, cache_directory):\n    """Explicit future Kaggle operation. Official package/source and weight are pinned."""\n    import clip\n    require(sha256(Path(clip.__file__).with_name("clip.py")) == CLIP_SOURCE_SHA256,\n            "Wrong OpenAI CLIP implementation")\n    url = clip.clip._MODELS["ViT-L/14"]\n    require(url.split("/")[-2] == CLIP_WEIGHT_SHA256, "Wrong companion weight identity")\n    model, preprocess = clip.load("ViT-L/14", device=device, jit=False,\n                                  download_root=str(cache_directory))\n    weight_path = Path(cache_directory) / "ViT-L-14.pt"\n    require(sha256(weight_path) == CLIP_WEIGHT_SHA256, "CLIP weight hash differs")\n    freeze(model)\n    vision = model.visual\n    require(int(vision.output_dim) == 768 and int(vision.input_resolution) == 224,\n            "Companion CLIP dimensions differ")\n    return vision, preprocess, dict(model="OpenAI ViT-L/14", commit=CLIP_COMMIT,\n        weight_sha256=CLIP_WEIGHT_SHA256, clip_source_sha256=CLIP_SOURCE_SHA256,\n        preprocess=repr(preprocess), historical_cache_revision="UNVERIFIED")\n\n\ndef preprocess_image(image, preprocess):\n    tensor = preprocess(image.convert("RGB"))\n    require(isinstance(tensor, torch.Tensor) and tuple(tensor.shape) == (3, 224, 224) and\n            tensor.dtype == torch.float32 and bool(torch.isfinite(tensor).all()),\n            "CLIP preprocessing must return finite float32 RGB [3,224,224]")\n    return tensor.unsqueeze(0)\n\n\ndef resolve_training_image(data_root, image_id, train_ids):\n    require(image_id in train_ids and Path(image_id).name == image_id and\n            "/" not in image_id and "\\\\" not in image_id, "Diagnostic image must be a saved training ID")\n    candidates = [p for p in Path(data_root).rglob(image_id) if p.is_file()]\n    require(len(candidates) == 1, "Training image missing or ambiguous")\n    require(candidates[0].resolve().is_relative_to(Path(data_root).resolve()), "Image escapes data root")\n    return candidates[0]\n\n\nclass FrozenImageToReport:\n    def __init__(self, vision, preprocess, decoder, prompts, vocabulary):\n        self.vision, self.preprocess, self.decoder = freeze(vision), preprocess, freeze(decoder)\n        require(decoder.architecture == ARCHITECTURE, "Historical decoder required")\n        require(tuple(prompts.shape) == (500, 768) and prompts.dtype == torch.float32 and\n                not prompts.requires_grad and bool(torch.isfinite(prompts).all()) and\n                bool((prompts.norm(dim=-1) > 0).all()), "Invalid raw fixed prompt tensor")\n        require(len(vocabulary) == 368, "Historical vocabulary required")\n        self.prompts, self.vocabulary = prompts.detach().clone(), list(vocabulary)\n        self.used = False\n\n    @torch.inference_mode()\n    def generate_one(self, image, max_length=160, progress=None):\n        require(not self.used, "Single-image diagnostic already attempted; no automatic retry")\n        require(2 <= max_length <= 160, "Generation exceeds historical positions")\n        self.used = True\n        log = progress or (lambda phase: None)\n        assert_frozen(self.vision)\n        assert_frozen(self.decoder)\n        device = next(self.decoder.parameters()).device\n        require(next(self.vision.parameters()).device == device, "Encoder/decoder device mismatch")\n        start = time.monotonic()\n        log("preprocess")\n        pixels = preprocess_image(image, self.preprocess).to(device)\n        log("image_encoder")\n        dtype = self.vision.conv1.weight.dtype\n        visual = self.vision(pixels.to(dtype=dtype)).float()\n        require(tuple(visual.shape) == (1, 768) and bool(torch.isfinite(visual).all()),\n                "CLIP output must be finite [1,768]")\n        log("prompt_enrichment_eq6_eq5")\n        prompts = self.prompts.to(device)\n        phi, attention = enrich(visual, prompts)\n        residual = visual + attention @ prompts\n        require(tuple(attention.shape) == (1, 500) and\n                torch.allclose(attention.sum(-1), torch.ones(1, device=device), atol=1e-5) and\n                torch.allclose(phi.norm(dim=-1), torch.ones(1, device=device), atol=1e-5),\n                "Attention/residual normalization failed")\n        # Use the existing historical greedy generation, with a hook to verify EVERY\n        # autoregressive forward is finite and visibly log bounded token progress.\n        forward_count = 0\n        def check_logits(module, inputs, output):\n            nonlocal forward_count\n            require(bool(torch.isfinite(output).all()), "Non-finite autoregressive logits")\n            forward_count += 1\n            if forward_count == 1 or forward_count % 20 == 0:\n                log("greedy_token_" + str(forward_count))\n        hook = self.decoder.register_forward_hook(check_logits)\n        try:\n            tokens = self.decoder.generate(phi, max_length=max_length).cpu()\n        finally:\n            hook.remove()\n        require(tuple(tokens.shape[:1]) == (1,) and 2 <= tokens.shape[1] <= max_length and\n                bool(((tokens >= 0) & (tokens < 368)).all()), "Invalid generated IDs")\n        assert_frozen(self.vision)\n        assert_frozen(self.decoder)\n        require(not self.prompts.requires_grad and torch.equal(prompts.cpu(), self.prompts.cpu()),\n                "Prompt tensor changed")\n        if device.type == "cuda":\n            torch.cuda.synchronize(device)\n        ids = tokens[0].tolist()\n        return dict(shapes=dict(pixels=list(pixels.shape), image_embedding=list(visual.shape),\n            prompts=list(prompts.shape), attention=list(attention.shape), residual=list(residual.shape),\n            decoder_condition=list(phi.shape), tokens=list(tokens.shape)),\n            norms=dict(image=visual.norm(dim=-1).cpu().tolist(),\n                prompt_min=float(prompts.norm(dim=-1).min()), prompt_max=float(prompts.norm(dim=-1).max()),\n                residual=residual.norm(dim=-1).cpu().tolist(), enriched=phi.norm(dim=-1).cpu().tolist()),\n            output_tokens=ids, report_text=decode_report(ids, self.vocabulary),\n            termination="EOS" if 2 in ids else "MAX_LENGTH", forward_count=forward_count,\n            device=str(device), elapsed_seconds=time.monotonic()-start,\n            gradients_disabled=True, all_modules_frozen=True, optimizer_created=False,\n            paired_supervision_used=False, test_references_accessed=False)\n', 'gate4_diagnostic_worker.py': '"""One future Kaggle training-image diagnostic, launched only by the bounded notebook."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport zipfile\nfrom importlib import metadata\nfrom pathlib import Path\n\nfrom zeromrg_gate1_runtime import require, sha256\nfrom zeromrg_gate4_runtime import (\n    CLIP_COMMIT, EPOCH90_SHA256, INFERENCE_FILES, FrozenImageToReport,\n    load_companion_clip, load_inference_artifacts, resolve_training_image, restore_epoch90,\n)\n\n\ndef log(phase):\n    print(json.dumps(dict(phase=phase, elapsed_seconds=round(time.monotonic()-START, 3))), flush=True)\n\n\ndef version(name):\n    try: return metadata.version(name)\n    except metadata.PackageNotFoundError: return None\n\n\ndef run_checked(args, timeout=300):\n    log("dependency_command")\n    subprocess.run(args, check=True, timeout=timeout)\n\n\ndef diagnostic(run_root, config):\n    import torch\n    require(config["max_images"] == 1 and config["max_tokens"] == 160 and\n            config["training_enabled"] is False and config["test_inference_enabled"] is False, "Diagnostic scope differs")\n    require(Path("/kaggle/input").is_dir() and run_root.resolve().is_relative_to(Path("/kaggle/working")),\n            "Kaggle-only diagnostic")\n    require(torch.cuda.is_available(), "Kaggle GPU required; no local fallback")\n    device = torch.device("cuda:0")\n    log("provided_environment_checks")\n    # These are the observed baseline stack. Changed images stop for review, not upgrade.\n    require(torch.__version__ == "2.11.0+cu128" and torch.version.cuda == "12.8", "Torch/CUDA drift; review required")\n    require(version("numpy") == "2.1.3" and version("Pillow") == "12.3.0", "NumPy/Pillow drift")\n    require(version("torchvision") is not None and version("torchvision").split("+")[0] == "0.26.0",\n            "Expected torchvision0.26.0 companion; do not auto-install Torch/torchvision")\n    import torchvision\n    from PIL import Image\n    # /tmp is ephemeral and excluded from Kaggle saved notebook outputs.\n    scratch = Path(tempfile.mkdtemp(prefix="zeromrg_gate4_",dir="/tmp"))\n    package_root = scratch / "inference_packages"\n    package_root.mkdir(exist_ok=False)\n    required = dict(ftfy="6.3.1", regex="2025.11.3", tqdm="4.67.1", wcwidth="0.9.2")\n    missing = [line for line in config["support_lock"].splitlines()\n               if version(line.split("==")[0]) != required[line.split("==")[0]]]\n    if missing:\n        lock = run_root / "support.lock"\n        lock.write_text("\\n".join(missing)+"\\n", encoding="utf-8")\n        run_checked([sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--require-hashes","--only-binary=:all:","--retries","0","--timeout","45",\n            "-r",str(lock)])\n    sys.path.insert(0,str(package_root))\n    for name, expected in required.items(): require(version(name) == expected, "Support package pin mismatch: "+name)\n    if version("clip") is None:\n        require(version("setuptools") is not None and version("wheel") is not None,\n                "Provided CLIP build tools missing; no implicit build-tool install")\n        run_checked([sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--no-build-isolation","--retries","0","--timeout","45",\n            "clip @ git+https://github.com/openai/CLIP.git@"+CLIP_COMMIT])\n    # The exact source hash is checked by load_companion_clip, even for provided clip.\n    log("verified_checkpoint_attachment")\n    candidates = list(Path("/kaggle/input").rglob(config["zip_name"]))\n    require(len(candidates) == 1, "Expected exactly one verified baseline ZIP attachment")\n    archive_path = candidates[0]\n    require(sha256(archive_path) == config["zip_sha256"], "Baseline ZIP differs")\n    artifact_root = scratch / "selected_inputs"\n    artifact_root.mkdir(exist_ok=False)\n    selected = ["recovery_best_epoch_090.pt"] + ["dependencies/"+name for name in INFERENCE_FILES]\n    with zipfile.ZipFile(archive_path) as archive:\n        require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP members")\n        for name in selected:\n            destination = artifact_root / name\n            require(destination.resolve().is_relative_to(artifact_root.resolve()), "Unsafe ZIP member")\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            # Only selected training inputs: reading to EOF verifies their ZIP CRC.\n            with archive.open(name) as source, destination.open("xb") as output:\n                for block in iter(lambda: source.read(1024*1024), b""): output.write(block)\n            require(sha256(destination) == config["selected_hashes"][name], "Selected input differs: "+name)\n    data_root = Path(config["data_root"])\n    require(data_root.is_dir() and data_root.resolve().is_relative_to(Path("/kaggle/input")), "COV data mount differs")\n    artifacts = load_inference_artifacts(artifact_root/"dependencies", config["dependency_hashes"])\n    require(hashlib.sha256(json.dumps(artifacts["prompt_ids"],separators=(",",":")).encode()).hexdigest() ==\n            config["prompt_ids_sha256"], "Prompt ordering fingerprint differs")\n    image_id = config["image_id"]\n    require(image_id == artifacts["train_ids"][0], "Fixed first saved training image differs")\n    image_path = resolve_training_image(data_root,image_id,artifacts["train_ids"])\n    image_sha256 = sha256(image_path)\n    log("strict_epoch90_restore")\n    decoder = restore_epoch90(artifact_root/"recovery_best_epoch_090.pt",device)\n    log("pinned_clip_load")\n    vision, preprocess, encoder = load_companion_clip(device,scratch/"clip_weights")\n    import clip\n    direct_url = metadata.distribution("clip").read_text("direct_url.json")\n    if direct_url:\n        vcs = json.loads(direct_url).get("vcs_info",{})\n        require(vcs.get("commit_id") == CLIP_COMMIT, "CLIP VCS provenance differs")\n    pipeline = FrozenImageToReport(vision,preprocess,decoder,artifacts["prompts"],artifacts["vocabulary"])\n    versions = {name:version(name) for name in\n                ("torch","torchvision","numpy","Pillow","clip","ftfy","regex","tqdm","wcwidth")}\n    before = dict(decoder=[p._version for p in decoder.parameters()],\n                  encoder=[p._version for p in vision.parameters()])\n    torch.cuda.synchronize()\n    inference_start = time.monotonic()\n    with Image.open(image_path) as image:\n        result = pipeline.generate_one(image,max_length=160,progress=log)\n    require(before == dict(decoder=[p._version for p in decoder.parameters()],\n                           encoder=[p._version for p in vision.parameters()]), "Parameter mutation detected")\n    result.update(status="PASS_SINGLE_TRAINING_IMAGE_DIAGNOSTIC",stage="zero_shot_image_to_report_diagnostic",\n        image_id=image_id,image_sha256=image_sha256,split="checkpoint_train",image_count=1,\n        checkpoint=dict(epoch=90,sha256=EPOCH90_SHA256,zip_sha256=config["zip_sha256"],\n                        reconstruction_bleu4=54.21140850351637,selection="available_physical_epoch90; historical85_missing"),\n        dependencies=versions,python=platform.python_version(),cuda=torch.version.cuda,\n        gpu=torch.cuda.get_device_name(0),encoder=encoder,artifact_hashes=artifacts["hashes"],\n        prompt_ids_sha256=config["prompt_ids_sha256"],config=config,\n        source_sha256={name:sha256(Path(__file__).parent/name) for name in\n                       ("zeromrg_gate1_runtime.py","zeromrg_gate4_runtime.py","gate4_diagnostic_worker.py")},\n        parameter_version_counters_unchanged=True,\n        total_worker_seconds=time.monotonic()-START,inference_seconds=time.monotonic()-inference_start,\n        scientific_limit="Historical encoder revision/cache alignment unverified; one image is not evaluation")\n    (run_root/"diagnostic.json").write_text(json.dumps(result,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n    log("diagnostic_saved")\n    print(json.dumps({k:result[k] for k in ("status","image_id","shapes","norms","output_tokens","report_text","termination")}),flush=True)\n\n\nif __name__ == "__main__":\n    START = time.monotonic()\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--run-root",required=True)\n    parser.add_argument("--config",required=True)\n    args = parser.parse_args()\n    destination = Path(args.run_root)\n    try:\n        diagnostic(destination,json.loads(Path(args.config).read_text(encoding="utf-8")))\n    except Exception as error:\n        # Never retry or change the architecture/environment automatically.\n        (destination/"failure.json").write_text(json.dumps(dict(status="FAILED",error_type=type(error).__name__,\n            error=str(error),elapsed_seconds=time.monotonic()-START),indent=2)+"\\n",encoding="utf-8")\n        log("failed_no_retry")\n        raise\n'}
SOURCE_HASHES = {'zeromrg_gate1_runtime.py': 'e897aa5e800ce18911744cc13e16b75b239477a4edb8bc1df0f8e9ab3f9e038a', 'zeromrg_gate4_runtime.py': 'c9117e4106168a907da3c52e7c99c0ca34655abf88b98e60842b241908c679d8', 'gate4_diagnostic_worker.py': 'e1ba00f6dd02a1ddfccd85c158283857d210b7bdee6148c4b528ffcf8dcc0812'}
CONFIG = {'run_id': 'ZM-010_GATE4_SINGLE_TRAIN_IMAGE_V1', 'image_id': 'PIIS0140673620301549_0%0.png', 'data_root': '/kaggle/input/datasets/inshasadaf114/dataset/COV-CTR with English reports', 'zip_name': 'recovery_export_epoch_090_54468b70.zip', 'zip_sha256': 'dec3cd3391e2c107a7c9ed6209997f00554cd47cb0940ca339a74446253680b1', 'selected_hashes': {'recovery_best_epoch_090.pt': '6166e36198737bfbe9e0c35f03f0b320c44e76f3e05068ef6299d74e9179c6b2', 'dependencies/train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'dependencies/prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'dependencies/decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'dependencies/mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}, 'dependency_hashes': {'train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}, 'prompt_ids_sha256': '2e1e001a0d2e45d35e8ee16522e4347bc9ee1bccb67ff9532d8c21cb9c2f65d0', 'support_lock': 'ftfy==6.3.1 --hash=sha256:7c70eb532015cd2f9adb53f101fb6c7945988d023a085d127d1573dc49dd0083\nregex==2025.11.3 --hash=sha256:4aecb6f461316adf9f1f0f6a4a1a3d79e045f9b71ec76055a791affa3b285850\ntqdm==4.67.1 --hash=sha256:26445eca388f82e72884e0d580d5464cd801a3ea01e63e5601bdff9ba6a48de2\nwcwidth==0.9.2 --hash=sha256:751bef0ab404b6a1dc028b56b4b85d46486be1c55833f80da533e42dc691f389\n', 'hard_timeout_seconds': 1800, 'max_images': 1, 'max_tokens': 160, 'training_enabled': False, 'test_inference_enabled': False, 'approval_scope': 'Prepared only; execution requires separate owner approval'}

In [ ]:
import os, queue, signal, subprocess, threading, time
require = lambda condition, message: None if condition else (_ for _ in ()).throw(ValueError(message))
require(not globals().get('_gate4_attempted', False), 'Already attempted; no automatic retry')
require(Path('/kaggle/input').is_dir(), 'Kaggle-only execution; local execution prohibited')
_gate4_attempted = True
RUN_ROOT = Path('/kaggle/working') / ('zeromrg_gate4_' + uuid.uuid4().hex[:12])
RUN_ROOT.mkdir(exist_ok=False)
for name, source in SOURCES.items():
    require(hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], 'Embedded source differs')
    (RUN_ROOT/name).write_text(source, encoding='utf-8')
(RUN_ROOT/'configuration.json').write_text(json.dumps(CONFIG,indent=2)+'\n',encoding='utf-8')
print('ONE training-image diagnostic; output:', RUN_ROOT, flush=True)
print('Hard worker limit:', CONFIG['hard_timeout_seconds'], 'seconds; engineering bound, not measured feasibility', flush=True)
started = time.monotonic()
command = [sys.executable, '-u', str(RUN_ROOT/'gate4_diagnostic_worker.py'),
           '--run-root', str(RUN_ROOT), '--config', str(RUN_ROOT/'configuration.json')]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
messages = queue.Queue()
def read_output():
    for line in process.stdout: messages.put(line)
reader = threading.Thread(target=read_output,daemon=True)
reader.start()
last_heartbeat = started
try:
    with (RUN_ROOT/'execution.log').open('x',encoding='utf-8') as output:
        while process.poll() is None or reader.is_alive() or not messages.empty():
            now = time.monotonic()
            if now-started > CONFIG['hard_timeout_seconds']:
                os.killpg(process.pid,signal.SIGKILL)
                process.wait(timeout=10)
                failure=dict(status='HARD_TIMEOUT',seconds=now-started,no_retry=True)
                (RUN_ROOT/'supervisor_failure.json').write_text(json.dumps(failure,indent=2)+'\n')
                raise TimeoutError('Gate4 worker exceeded hard bound; no retry')
            try:
                line=messages.get(timeout=1)
                output.write(line); output.flush(); print(line,end='',flush=True)
            except queue.Empty:
                pass
            if now-last_heartbeat >= 30:
                print('Supervisor alive; elapsed',round(now-started,1),'seconds',flush=True)
                last_heartbeat=now
    require(process.returncode == 0, 'Diagnostic failed; inspect execution.log/failure.json; no retry')
    result=json.loads((RUN_ROOT/'diagnostic.json').read_text(encoding='utf-8'))
    require(result['image_count'] == 1 and result['split'] == 'checkpoint_train' and
            not result['test_references_accessed'] and not result['optimizer_created'], 'Diagnostic scope differs')
    print('Prepared diagnostic result:', RUN_ROOT/'diagnostic.json',flush=True)
finally:
    if process.poll() is None:
        os.killpg(process.pid,signal.SIGKILL)
        process.wait(timeout=10)